In [1]:
import sys
sys.path.append('../..')
import numpy as np
from cpm.generators.wrapper import MetaSignalDetectionWrapper
from cpm.models.signal_detection import MetaSignalDetectionHelper, MetaSignalDetectionModel
from cpm.optimisation import minimise, DifferentialEvolution, FminBound, Bads
from cpm.applications.MetacognitionSDT import metacognitionSDT_initparams

In [2]:
nR_S1 = [36, 24, 17, 20, 10, 12, 9, 2]
nR_S2 = [1, 4, 10, 11, 19, 18, 28, 39]
nbins = 4

binned_data = {
    'nR_S1': np.array(nR_S1),
    'nR_S2': np.array(nR_S2),
}

In [3]:
params, d1, c1, data_pandas = metacognitionSDT_initparams(binned_data=binned_data, nbins=4)
data_pandas["ppt"] = 0

print(params.t2c1.prior.rvs())

[[20 17 24 36]
 [10 12  9  2]
 [19 18 28 39]
 [11 10  4  1]]
   nbins                                           observed
0      4  [[20, 17, 24, 36], [10, 12, 9, 2], [19, 18, 28...
[-3.20710381 -1.09323429  0.78733277 -0.74493549  2.93958763  0.27519038]


In [7]:
def model(parameters):

    print(parameters.t1c1.value)
    print(parameters.t2c1.value)
    parameters.t2c1.value = np.sort(parameters.t2c1.value)
    print(parameters.t2c1.value)
    msd = MetaSignalDetectionModel(nbins=nbins, d1=parameters.d1, t1c1=parameters.t1c1, meta_d1=parameters.meta_d1, t2c1=parameters.t2c1)
    output = {
        "nbins": nbins,
        "dependent": msd.t2_probs(),
    }
    return output

print(model(params))

[1.13471025 1.19737796 1.20696745 1.27441959 1.34222247 1.38376092]
[1.38376092 1.34222247 1.27441959 1.20696745 1.19737796 1.13471025]
{'nbins': 4, 'dependent': array([[ 1.02338134e+00, -6.11299041e-05, -1.19826652e-04,
        -2.32003871e-02],
       [ 1.41085563e+00, -4.35331024e-03, -7.85263626e-03,
        -3.98649688e-01],
       [ 8.55932535e-01, -2.93268371e-03, -2.03639985e-02,
         1.67364147e-01],
       [ 9.70244130e-01, -1.00496186e-03, -7.37646847e-03,
         3.81373007e-02]])}


In [8]:
wrapper = MetaSignalDetectionWrapper(model=model, data=binned_data, parameters=params)

In [9]:
fit = Bads(
    model=wrapper,  # Wrapper class with the model we specified from before
    data=data_pandas.groupby("ppt"),
    minimisation=minimise.LogLikelihood.multinomial,
    parallel=False,
    prior=False,
    ppt_identifier="ppt",
    number_of_starts=5,
)

fit.optimise()

Starting optimization 1/5 from [  0.87920695 -33.85765406 -57.71002949  -0.88430169  39.60540397
  21.47269323  47.82758124]
guess [  0.87920695 -33.85765406 -57.71002949  -0.88430169  39.60540397
  21.47269323  47.82758124]
bounds [0, np.int64(-100), np.int64(-100), np.int64(-100), np.int64(0), np.int64(0), np.int64(0)]
bounds [1, np.int64(0), np.int64(0), np.int64(0), np.int64(100), np.int64(100), np.int64(100)]
bads:TooCloseBounds: For each variable, hard and plausible bounds should not be too close. Moving plausible bounds.
[-33.87021484 -57.69941406  -0.8796875   39.62041016  21.49267578
  47.80712891]
[ 47.80712891  39.62041016  21.49267578  -0.8796875  -33.87021484
 -57.69941406]
[-33.87021484 -57.69941406  -0.8796875   39.62041016  21.49267578
  47.80712891]
[ 47.80712891  39.62041016  21.49267578  -0.8796875  -33.87021484
 -57.69941406]
Beginning optimization of a DETERMINISTIC objective function

 Iteration    f-count         f(x)           MeshScale          Method          

[-50.19492187 -96.0015625  -35.67324219   5.11923828  52.48525391
  35.03974609]
[ 52.48525391  35.03974609   5.11923828 -35.67324219 -50.19492187
 -96.0015625 ]
[-21.24902344 -56.38369141 -39.18183594  14.134375    92.73662109
  98.73046875]
[ 98.73046875  92.73662109  14.134375   -21.24902344 -39.18183594
 -56.38369141]
[-48.19697266 -25.48857422 -54.28828125  49.2203125   78.50732422
  54.72685547]
[ 78.50732422  54.72685547  49.2203125  -25.48857422 -48.19697266
 -54.28828125]
[-80.26162109 -14.42675781 -70.271875    32.40830078  63.59580078
  16.27851562]
[ 63.59580078  32.40830078  16.27851562 -14.42675781 -70.271875
 -80.26162109]
[ -1.659375    -6.62988281 -94.39345703  79.23828125  29.48447266
  81.13876953]
[ 81.13876953  79.23828125  29.48447266  -1.659375    -6.62988281
 -94.39345703]
[-70.75917969 -45.85791016 -80.74892578  89.03310547  12.81865234
  41.71582031]
[ 89.03310547  41.71582031  12.81865234 -45.85791016 -70.75917969
 -80.74892578]
[-96.82998047 -64.08310547 -13

AttributeError: 'float' object has no attribute 'item'

In [ ]:
from scipy.stats import multinomial

n = np.array([[1,2,3,4],[1,2,3,4]])
p = np.array([[0.2,0.3,0.1,0.4],[0.1,0.2,0.3,0.4]])

print(n.shape, p.shape)

np.sum([multinomial.logpmf(n[i], 10.0, p[i]) for i in range(p.shape[0])])